In [ ]:
!pip install numpy pandas matplotlib scipy yfinance arch statsmodels -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Chart style
BLUE = '#1A3A6E'; RED = '#DC3545'; GREEN = '#2E7D32'
ORANGE = '#E67E22'; GRAY = '#666666'; PURPLE = '#8E44AD'

plt.rcParams.update({
    'figure.facecolor': 'none', 'axes.facecolor': 'none',
    'savefig.facecolor': 'none', 'savefig.transparent': True,
    'axes.grid': False, 'font.size': 10, 'axes.labelsize': 11,
    'axes.titlesize': 12, 'legend.fontsize': 9, 'xtick.labelsize': 9,
    'ytick.labelsize': 9, 'axes.spines.top': False, 'axes.spines.right': False,
    'lines.linewidth': 1.0, 'axes.linewidth': 0.6,
    'legend.facecolor': 'none', 'legend.framealpha': 0, 'legend.edgecolor': 'none',
    'figure.dpi': 150,
})

def save_chart(fig, name):
    fig.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True, dpi=150)
    fig.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved: {name}')

def legend_bottom(ax, ncol=2, y=-0.18):
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)

In [ ]:
import yfinance as yf
from arch import arch_model
from scipy.optimize import minimize

tickers = ['^GSPC', '^N225']
names = ['S&P 500', 'Nikkei 225']
data = {}
for tick, name in zip(tickers, names):
    d = yf.download(tick, start='2010-01-01', end='2025-12-31', progress=False)
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.get_level_values(0)
    data[name] = (d['Close'].pct_change() * 100).dropna()

returns_df = pd.DataFrame(data).dropna()

# Univariate GJR-GARCH for asymmetric effects
std_resid = pd.DataFrame(index=returns_df.index)
for name in names:
    m = arch_model(returns_df[name], vol='Garch', p=1, o=1, q=1, dist='t')
    r = m.fit(disp='off')
    std_resid[name] = r.resid / r.conditional_volatility

z = std_resid.values
T, k = z.shape
Q_bar = z.T @ z / T
print(f"Observations: {T}, Assets: {k}")

In [ ]:
# ============================================================
# Standard DCC
# ============================================================
def dcc_compute(params, z, Q_bar):
    a, b = params
    T, k = z.shape
    Q_t = Q_bar.copy()
    rho = np.zeros(T)
    for t in range(1, T):
        Q_t = (1-a-b)*Q_bar + a*np.outer(z[t-1], z[t-1]) + b*Q_t
        d = np.sqrt(np.diag(Q_t))
        rho[t] = Q_t[0,1] / (d[0]*d[1])
    return rho

def dcc_nll(params, z, Q_bar):
    a, b = params
    if a<0 or b<0 or a+b>=0.999: return 1e10
    T, k = z.shape; Q_t = Q_bar.copy(); nll = 0
    for t in range(1, T):
        Q_t = (1-a-b)*Q_bar + a*np.outer(z[t-1], z[t-1]) + b*Q_t
        d = np.sqrt(np.diag(Q_t)); R = Q_t/np.outer(d,d)
        try:
            s, ld = np.linalg.slogdet(R)
            if s<=0: return 1e10
            nll += 0.5*(ld + z[t]@np.linalg.inv(R)@z[t] - z[t]@z[t])
        except: return 1e10
    return nll

res_dcc = minimize(dcc_nll, [0.02, 0.95], args=(z, Q_bar), method='Nelder-Mead')
a_dcc, b_dcc = res_dcc.x
rho_dcc = dcc_compute([a_dcc, b_dcc], z, Q_bar)
print(f"DCC:  a={a_dcc:.4f}, b={b_dcc:.4f}")

In [ ]:
# ============================================================
# Asymmetric DCC (aDCC, Cappiello et al. 2006)
# Q_t = (1-a-b)*Q_bar - g*N_bar + a*z'z + b*Q_{t-1} + g*n'n
# where n_t = z_t * I(z_t < 0)
# ============================================================
n = z * (z < 0)  # negative shocks only
N_bar = n.T @ n / T

def adcc_nll(params, z, Q_bar, n, N_bar):
    a, b, g = params
    if a<0 or b<0 or g<0 or a+b+0.5*g>=0.999: return 1e10
    T, k = z.shape; Q_t = Q_bar.copy(); nll = 0
    for t in range(1, T):
        Q_t = (1-a-b)*Q_bar - g*N_bar + a*np.outer(z[t-1], z[t-1]) + b*Q_t + g*np.outer(n[t-1], n[t-1])
        d = np.sqrt(np.maximum(np.diag(Q_t), 1e-10)); R = Q_t/np.outer(d,d)
        R = np.clip(R, -0.9999, 0.9999); np.fill_diagonal(R, 1.0)
        try:
            s, ld = np.linalg.slogdet(R)
            if s<=0: return 1e10
            nll += 0.5*(ld + z[t]@np.linalg.inv(R)@z[t] - z[t]@z[t])
        except: return 1e10
    return nll

def adcc_compute(params, z, Q_bar, n, N_bar):
    a, b, g = params
    T, k = z.shape; Q_t = Q_bar.copy(); rho = np.zeros(T)
    for t in range(1, T):
        Q_t = (1-a-b)*Q_bar - g*N_bar + a*np.outer(z[t-1], z[t-1]) + b*Q_t + g*np.outer(n[t-1], n[t-1])
        d = np.sqrt(np.maximum(np.diag(Q_t), 1e-10))
        rho[t] = Q_t[0,1]/(d[0]*d[1])
    return rho

res_adcc = minimize(adcc_nll, [0.02, 0.93, 0.02], args=(z, Q_bar, n, N_bar), method='Nelder-Mead')
a_a, b_a, g_a = res_adcc.x
rho_adcc = adcc_compute([a_a, b_a, g_a], z, Q_bar, n, N_bar)
print(f"aDCC: a={a_a:.4f}, b={b_a:.4f}, g={g_a:.4f}")
print(f"g > 0 confirms asymmetric correlation increase during joint downturns")

In [ ]:
# Compare DCC vs aDCC
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

ax = axes[0]
ax.plot(returns_df.index, rho_dcc, color=BLUE, lw=0.5, label='DCC')
ax.plot(returns_df.index, rho_adcc, color=RED, lw=0.5, alpha=0.7, label='aDCC')
ax.axhline(Q_bar[0,1], color='black', ls='--', lw=1, alpha=0.5)
ax.set_ylabel('ρ_t')
ax.set_title('DCC vs aDCC: S&P 500 — Nikkei 225', fontweight='bold')
legend_bottom(ax, ncol=2, y=-0.12)

ax = axes[1]
diff = rho_adcc - rho_dcc
ax.fill_between(returns_df.index, diff, 0, where=diff>0, alpha=0.5, color=RED, label='aDCC > DCC')
ax.fill_between(returns_df.index, diff, 0, where=diff<=0, alpha=0.5, color=GREEN, label='aDCC < DCC')
ax.axhline(0, color='black', lw=0.5)
ax.set_ylabel('ρ_aDCC − ρ_DCC')
ax.set_title('Difference: aDCC captures higher correlation during crises', fontweight='bold')
legend_bottom(ax, ncol=2, y=-0.15)

plt.tight_layout()
save_chart(fig, 'garch_dcc_extensions')
plt.show()